In [1]:
from edsl import (
    QuestionFreeText,
    QuestionMultipleChoice,
    QuestionNumerical,
    Scenario,
    ScenarioList,
    Survey,
    Agent,
    AgentList,
    Model,
)
import os
import random
import time
import ast
import csv
import goodfire

In [2]:
# Initialize Goodfire client
client = goodfire.Client(os.getenv("GOODFIRE_API_KEY"))
# Create a Goodfire variant object instead of just using a string
base_variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

In [3]:
# Base Experiment Class
class SocialExperiment:
    def __init__(self, name, agents, model):
        self.name = name
        self.agents = agents
        self.model = model
        self.questions = []
        self.variant = goodfire.Variant(
            "meta-llama/Llama-3.3-70B-Instruct"
        )  # Initialize with base variant

    def setup(self):
        """Sets up the experiment by defining questions and scenarios."""
        raise NotImplementedError

    def run(self, intervention=True):
        """Runs the experiment, optionally applying Goodfire interventions."""
        if intervention:
            self.model.parameters["controller"] = self.get_intervention()
            # For feature inspection, we need to apply the same controller to our variant
            controller_json = self.get_intervention()
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

            # Extract and apply features from controller JSON
            if "interventions" in controller_json:
                for intervention in controller_json["interventions"]:
                    if (
                        "features" in intervention
                        and "features" in intervention["features"]
                    ):
                        for feature_data in intervention["features"]["features"]:
                            # Create a feature object using feature search instead of direct creation
                            # This avoids needing to provide the index_in_sae
                            features = client.features.search(
                                feature_data["label"], model=self.variant, top_k=1
                            )
                            if features:
                                # Apply the feature with its value
                                self.variant.set(features[0], intervention["value"])
        else:
            self.model.parameters["controller"] = {}
            # Reset variant to base
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

        survey = Survey(self.questions)
        results = survey.by(self.agents).by(self.model).run()
        return results

    def get_intervention(self):
        """Defines the Goodfire interventions for the experiment."""
        raise NotImplementedError

    def save_results(self, results, filename):
        """Saves results to a CSV file."""
        results.to_csv(filename)

    def print_results(self, results, experiment_name):
        """Prints formatted results for the experiment."""
        print(f"\n{experiment_name} Results:")
        print("--------------------")
        for question in self.questions:
            question_name = question.question_name
            ans = results.select(f"answer.{question_name}")
            com = results.select(f"comment.{question_name}_comment")
            print(f"\nQuestion: {question_name}")
            print(f"Answer: {ans}\nComment: {com}")

    def get_feature_activations(self, results, scenario_name, use_base_variant=False):
        """Analyze feature activation for results"""
        print(f"\n--- {scenario_name} Feature Activation ---")

        # Use base variant if requested (for baseline analysis)
        inspect_variant = base_variant if use_base_variant else self.variant

        # Get all question names from the result
        question_names = []
        for column in results.columns:
            if column.startswith("answer."):
                question_name = column.split(".")[1]
                if question_name not in question_names:
                    question_names.append(question_name)

        # Process each question
        for question_name in question_names:
            print(f"\nQuestion: {question_name}")

            # Get the user prompts and model responses
            try:
                # Select the relevant columns
                user_prompts = results.select(f"prompt.{question_name}_user_prompt")
                model_responses = results.select(
                    f"generated_tokens.{question_name}_generated_tokens"
                )

                # Convert to lists
                user_prompt_list = user_prompts.to_list()
                model_response_list = model_responses.to_list()

                # Check if we have data
                if user_prompt_list and model_response_list:
                    # Process a few examples
                    for i, (user_prompt, model_response) in enumerate(
                        zip(user_prompt_list, model_response_list)
                    ):
                        if i >= 3:  # Limit to 3 examples to avoid too much output
                            break

                        if user_prompt and model_response:
                            # Convert the user prompt from dict to string if necessary
                            if isinstance(user_prompt, dict) and "text" in user_prompt:
                                user_prompt = user_prompt["text"]

                            print(f"\nResponse {i+1}:")
                            print(
                                f"User prompt (truncated): {str(user_prompt)[:100]}..."
                            )

                            # Inspect feature activation
                            try:
                                inspector = client.features.inspect(
                                    [
                                        {"role": "user", "content": str(user_prompt)},
                                        {
                                            "role": "assistant",
                                            "content": str(model_response),
                                        },
                                    ],
                                    model=inspect_variant,  # Use the appropriate variant
                                )

                                # Display top activated features
                                print("Top activated features:")
                                for activation in inspector.top(k=10):
                                    print(
                                        f"{activation.feature.label}: {activation.activation}"
                                    )
                            except Exception as e:
                                print(f"Error analyzing features: {e}")
                else:
                    print(f"No data found for question {question_name}")
            except Exception as e:
                print(f"Error processing data for question {question_name}: {e}")

In [4]:
# Trust Game Experiment with controlled sent amount
class TrustGame(SocialExperiment):
    def __init__(self, name, agents, model, fixed_sent_amount=None):
        super().__init__(name, agents, model)
        self.fixed_sent_amount = fixed_sent_amount
        
    def setup(self):
        self.questions = []
        
        if self.fixed_sent_amount is None:
            # Player A (Sender) Question - only needed if we're not forcing a specific amount
            send_question_text = (
                "TRUST GAME EXPERIMENT - PLAYER A (SENDER)\n\n"
                "You are participating in a Trust Game with another participant. Here's how it works:\n\n"
                "1. You are the 'sender' (Player A) and are given 100 tokens.\n"
                "2. You can choose to send any amount (from 0 to 100) to the 'receiver' (Player B).\n"
                "3. Any amount you send will be TRIPLED before it reaches the receiver.\n"
                "4. The receiver then decides how much to return to you (they can return any amount from 0 to what they received).\n"
                "5. You keep any tokens you don't send plus any tokens returned to you.\n\n"
                "You have been endowed with 100 tokens.\n"
                "How much would you like to send to your partner (Player B)? "
            )

            q_send = QuestionNumerical(
                question_name="trust_send",
                question_text=send_question_text,
                min_value=0,
                max_value=100,
            )
            self.questions.append(q_send)

        # Player B (Receiver) Question
        return_question_text = (
            "TRUST GAME EXPERIMENT - PLAYER B (RECEIVER)\n\n"
            "You are participating in a Trust Game with another participant. Here's how it works:\n\n"
            "1. The 'sender' (Player A) was given 100 tokens and chose to send some portion to you.\n"
            "2. The amount sent to you has been TRIPLED.\n"
            "3. You now need to decide how much to return to Player A.\n"
            "4. You can return any amount from 0 up to the tripled amount you received.\n"
            "5. You keep any tokens you don't return.\n\n"
            "Player A sent you {{ SENT_AMOUNT }} tokens, which has been tripled to {{ TRIPLED_AMOUNT }} tokens.\n\n"
            "How much would you like to return to Player A? "
        )

        q_return = QuestionNumerical(
            question_name="trust_return",
            question_text=return_question_text,
            min_value=0,
            max_value=300,  # Maximum possible tripled amount
        )
        self.questions.append(q_return)

    def get_intervention(self):
        return {
            "interventions": [
                {
                    "mode": "nudge",
                    "features": {
                        "features": [
                            {
                                "uuid": "4e2c9b8b769d4ceeac837509ffc530e9",
                                "label": "Expressions of trust in intimate or vulnerable contexts",
                                "index_in_sae": 38558,
                                "max_activation_strength": 1,
                            }
                        ]
                    },
                    "value": 0.4,
                },
                {
                    "mode": "nudge",
                    "features": {
                        "features": [
                            {
                                "uuid": "a4b0c6bd745744e6a64035b6c7c44e1b",
                                "label": "Expressions of trustworthiness and dependability",
                                "index_in_sae": 17623,
                                "max_activation_strength": 1,
                            }
                        ]
                    },
                    "value": 0.3,
                },
                {
                    "mode": "nudge",
                    "features": {
                        "features": [
                            {
                                "uuid": "c50be65067cc4f808ec7531c24468cbf",
                                "label": "Expressions of trust and confidence in relationships or beliefs",
                                "index_in_sae": 39359,
                                "max_activation_strength": 1,
                            }
                        ]
                    },
                    "value": 0.3,
                },
                {
                    "mode": "nudge",
                    "features": {
                        "features": [
                            {
                                "uuid": "1194a247aee14519a8ba41282c35b13a",
                                "label": "Trust and trustworthiness in relationships",
                                "index_in_sae": 11444,
                                "max_activation_strength": 1,
                            }
                        ]
                    },
                    "value": 0.4,
                },
            ],
            "scopes": [],
            "name": "controller__43997065",
            "nonzero_strength_threshold": None,
            "min_nudge_entropy": None,
            "max_nudge_entropy": None,
        }

    def run(self, intervention=True):
        """Overriding the run method to handle both parts of the trust game"""
        if intervention:
            self.model.parameters["controller"] = self.get_intervention()
        else:
            self.model.parameters["controller"] = {}
            # Reset variant to base
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

        # If fixed_sent_amount is provided, skip Player A question
        if self.fixed_sent_amount is not None:
            sent_amount = self.fixed_sent_amount
            tripled_amount = sent_amount * 3
            print(f"Using fixed sent amount: {sent_amount}, Tripled amount: {tripled_amount}")
            
            # No need to run sender question - use the directly provided amount
            send_results = None
        else:
            # Step 1: Run the sender question
            sender_question = Survey([self.questions[0]])
            send_results = sender_question.by(self.agents).by(self.model).run()

            # Step 2: Extract the amount sent
            try:
                # Extract the first send amount (simplifying by using just one value)
                sent_data = send_results.select("answer.trust_send")[0]
                if isinstance(sent_data, dict):
                    sent_amount = float(sent_data["answer.trust_send"][0])
                else:
                    sent_amount = float(sent_data)
            except Exception as e:
                print(f"Error extracting send amount: {e}")
                sent_amount = 50  # Default if extraction fails

            # Get tripled amount from sent amount
            tripled_amount = sent_amount * 3
            print("--------------------------------")
            print(f"Sent amount: {sent_amount}, Tripled amount: {tripled_amount}")
            print("--------------------------------")

        # Create a scenario for the values
        scenario = Scenario(
            {"SENT_AMOUNT": sent_amount, "TRIPLED_AMOUNT": tripled_amount}
        )

        # Use ScenarioList to apply the scenario
        scenarios = ScenarioList([scenario])

        # Update the receiver question's max value
        return_q = self.questions[-1]  # Use last question (only question if fixed_sent_amount provided)
        return_q.max_value = tripled_amount

        # Run the receiver question with the scenario
        receiver_question = Survey([return_q])
        return_results = (
            receiver_question.by(self.agents).by(self.model).by(scenarios).run()
        )

        # Return the results
        return {
            "send_results": send_results, 
            "return_results": return_results,
            "sent_amount": sent_amount,
            "tripled_amount": tripled_amount
        }

    def get_feature_activations_for_trust(
        self, results_dict, scenario_name, use_base_variant=False
    ):
        """Special version of feature activation analysis for trust game results"""
        print(f"\n=== {scenario_name} Feature Activation Analysis ===")

        # Check if send_results exists (it won't exist for fixed_sent_amount)
        if results_dict["send_results"] is not None:
            print("\n--- Player A (Sender) ---")
            self.get_feature_activations(
                results_dict["send_results"], f"{scenario_name} Sender", use_base_variant
            )

        # Analyze receiver results
        print("\n--- Player B (Receiver) ---")
        self.get_feature_activations(
            results_dict["return_results"],
            f"{scenario_name} Receiver",
            use_base_variant,
        )

In [5]:
# Agent instructions for better context
agent_instructions = """You are participating in an economic experiment at a research lab. Please behave as a typical participant would:

1. Read all instructions carefully
2. Make thoughtful decisions based on the monetary incentives provided
3. Consider the consequences of your choices for both yourself and other participants
4. Answer honestly and to the best of your ability

Your decisions will have monetary consequences in the experiment, so please consider them carefully."""

dummy_agents = AgentList(
    [
        Agent(
            name="Agent_1",
            traits={"subject_id": "A1"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_2",
            traits={"subject_id": "A2"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_3",
            traits={"subject_id": "A3"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_4",
            traits={"subject_id": "A4"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_5",
            traits={"subject_id": "A5"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_6",
            traits={"subject_id": "A6"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_7",
            traits={"subject_id": "A7"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_8",
            traits={"subject_id": "A8"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_9",
            traits={"subject_id": "A9"},
            instruction=agent_instructions,
        ),
        Agent(
            name="Agent_10",
            traits={"subject_id": "A10"},
            instruction=agent_instructions,
        ),
    ]
)

model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")


In [6]:
# Function to run trust game with different sent amounts
def run_trust_game_experiments(agents, model, start=10, end=100, step=10):
    results = []

    # First run a standard trust game where agents decide how much to send
    print("\n--- Running Trust Game with agents deciding sent amounts ---")
    experiment_name = "Trust Game (Agent Decision)"
    experiment = TrustGame(experiment_name, agents, model, fixed_sent_amount=None)
    experiment.setup()

    # Run with and without intervention
    print(f"\nRunning {experiment_name} without intervention...")
    baseline_results = experiment.run(intervention=False)

    print(f"\nRunning {experiment_name} with intervention...")
    intervention_results = experiment.run(intervention=True)

    # Extract and calculate the average sent amounts
    baseline_sent_amounts = []
    intervention_sent_amounts = []

    try:
        if baseline_results["send_results"] is not None:
            baseline_sent = (
                baseline_results["send_results"].select("answer.trust_send").to_list()
            )
            baseline_sent_amounts = [
                float(x) if isinstance(x, (int, float, str)) else float(x[0])
                for x in baseline_sent
            ]

        if intervention_results["send_results"] is not None:
            intervention_sent = (
                intervention_results["send_results"]
                .select("answer.trust_send")
                .to_list()
            )
            intervention_sent_amounts = [
                float(x) if isinstance(x, (int, float, str)) else float(x[0])
                for x in intervention_sent
            ]

        avg_baseline_sent = (
            sum(baseline_sent_amounts) / len(baseline_sent_amounts)
            if baseline_sent_amounts
            else 0
        )
        avg_intervention_sent = (
            sum(intervention_sent_amounts) / len(intervention_sent_amounts)
            if intervention_sent_amounts
            else 0
        )

        print(f"\nAverage amounts sent by agents:")
        print(f"Baseline - Avg sent: {avg_baseline_sent:.2f}")
        print(f"Intervention - Avg sent: {avg_intervention_sent:.2f}")

        # Save agent decision results (summary)
        agent_decision_file = os.path.join(results_dir, "trust_game_agent_decision.csv")
        with open(agent_decision_file, "w", newline="") as csvfile:
            fieldnames = ["avg_sent_baseline", "avg_sent_intervention"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerow(
                {
                    "avg_sent_baseline": avg_baseline_sent,
                    "avg_sent_intervention": avg_intervention_sent,
                }
            )

        # Save detailed agent-by-agent sent amounts
        agent_detail_file = os.path.join(results_dir, "trust_game_agent_detail.csv")
        with open(agent_detail_file, "w", newline="") as csvfile:
            # Create fieldnames for the CSV
            fieldnames = ["agent_id", "baseline_sent", "intervention_sent"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

            # Get agent IDs from the baseline results
            if baseline_results["send_results"] is not None:
                try:
                    agent_ids = (
                        baseline_results["send_results"].select("agent.name").to_list()
                    )
                except:
                    # If we can't get agent names, just use numbered IDs
                    agent_ids = [
                        f"Agent_{i+1}" for i in range(len(baseline_sent_amounts))
                    ]
            else:
                agent_ids = [f"Agent_{i+1}" for i in range(len(baseline_sent_amounts))]

            # Write each agent's data
            for i in range(
                max(len(baseline_sent_amounts), len(intervention_sent_amounts))
            ):
                agent_id = agent_ids[i] if i < len(agent_ids) else f"Agent_{i+1}"
                baseline_val = (
                    baseline_sent_amounts[i]
                    if i < len(baseline_sent_amounts)
                    else "N/A"
                )
                intervention_val = (
                    intervention_sent_amounts[i]
                    if i < len(intervention_sent_amounts)
                    else "N/A"
                )

                writer.writerow(
                    {
                        "agent_id": agent_id,
                        "baseline_sent": baseline_val,
                        "intervention_sent": intervention_val,
                    }
                )

        print(f"Detailed agent-by-agent sent amounts saved to {agent_detail_file}")

    except Exception as e:
        print(f"Error processing agent decision results: {e}")
        agent_decision_results = {
            "avg_sent_baseline": 0,
            "avg_sent_intervention": 0,
            "baseline_sent_amounts": [],
            "intervention_sent_amounts": [],
        }

    # Continue with the fixed sent amount experiments as before
    for sent_amount in range(start, end + step, step):
        print(f"\n--- Running Trust Game with sent amount: {sent_amount} tokens ---")

        # Create experiment with current sent amount
        experiment_name = f"Trust Game (Sent: {sent_amount})"
        experiment = TrustGame(
            experiment_name, agents, model, fixed_sent_amount=sent_amount
        )
        experiment.setup()

        # Run with and without intervention
        print(f"\nRunning {experiment_name} without intervention...")
        baseline_results = experiment.run(intervention=False)

        print(f"\nRunning {experiment_name} with intervention...")
        intervention_results = experiment.run(intervention=True)

        # Save results to CSV
        results_dir = "edsl_games/trust_game/results"
        os.makedirs(results_dir, exist_ok=True)

        baseline_filename = os.path.join(
            results_dir, f"trust_game_baseline_sent_{sent_amount}.csv"
        )
        intervention_filename = os.path.join(
            results_dir, f"trust_game_intervention_sent_{sent_amount}.csv"
        )

        experiment.save_results(baseline_results["return_results"], baseline_filename)
        experiment.save_results(
            intervention_results["return_results"], intervention_filename
        )

        # Extract return amounts
        try:
            baseline_returns = (
                baseline_results["return_results"]
                .select("answer.trust_return")
                .to_list()
            )
            baseline_returns = [
                float(x) if isinstance(x, (int, float, str)) else float(x[0])
                for x in baseline_returns
            ]

            intervention_returns = (
                intervention_results["return_results"]
                .select("answer.trust_return")
                .to_list()
            )
            intervention_returns = [
                float(x) if isinstance(x, (int, float, str)) else float(x[0])
                for x in intervention_returns
            ]

            avg_baseline_return = (
                sum(baseline_returns) / len(baseline_returns) if baseline_returns else 0
            )
            avg_intervention_return = (
                sum(intervention_returns) / len(intervention_returns)
                if intervention_returns
                else 0
            )

            # Calculate return ratios (returned amount / tripled amount)
            tripled_amount = sent_amount * 3
            return_ratio_baseline = (
                avg_baseline_return / tripled_amount if tripled_amount > 0 else 0
            )
            return_ratio_intervention = (
                avg_intervention_return / tripled_amount if tripled_amount > 0 else 0
            )

            # Print current results
            print(f"\nSent amount: {sent_amount}, Tripled amount: {tripled_amount}")
            print(
                f"Baseline - Avg return: {avg_baseline_return:.2f}, Return ratio: {return_ratio_baseline:.2f}"
            )
            print(
                f"Intervention - Avg return: {avg_intervention_return:.2f}, Return ratio: {return_ratio_intervention:.2f}"
            )

            # Store results for summary
            results.append(
                {
                    "sent_amount": sent_amount,
                    "tripled_amount": tripled_amount,
                    "avg_return_baseline": avg_baseline_return,
                    "avg_return_intervention": avg_intervention_return,
                    "return_ratio_baseline": return_ratio_baseline,
                    "return_ratio_intervention": return_ratio_intervention,
                }
            )
        except Exception as e:
            print(f"Error processing results for sent amount {sent_amount}: {e}")
        # Run baseline experiment
        print("\nRunning analysis experiment without intervention...")
        results_baseline = experiment.run(intervention=False)

        # Run intervention experiment
        print("\nRunning analysis experiment with intervention...")
        results_intervention = experiment.run(intervention=True)

        # Feature activation analysis
        # print("\n=== Feature Activation Analysis ===")

        # # Analyze baseline feature activation
        # experiment.get_feature_activations_for_trust(
        #     results_baseline, "Baseline", use_base_variant=True
        # )

        # # Analyze intervention feature activation
        # experiment.get_feature_activations_for_trust(
        #     results_intervention, "Intervention", use_base_variant=False
        # )

    # Write summary to CSV
    summary_file = os.path.join(results_dir, "trust_game_experiment_summary.csv")
    with open(summary_file, "w", newline="") as csvfile:
        fieldnames = [
            "sent_amount",
            "tripled_amount",
            "avg_return_baseline",
            "avg_return_intervention",
            "return_ratio_baseline",
            "return_ratio_intervention",
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        # First write the agent decision row with special formatting
        writer.writerow(
            {
                "sent_amount": "Agent Decision",
                "tripled_amount": f"{agent_decision_results['avg_sent_baseline']*3:.2f}/{agent_decision_results['avg_sent_intervention']*3:.2f}",
                "avg_return_baseline": "",
                "avg_return_intervention": "",
                "return_ratio_baseline": "",
                "return_ratio_intervention": "",
            }
        )

        # Then write all the fixed amount rows
        for row in results:
            writer.writerow(row)

    print(f"\nResults summary saved to {summary_file}")

    # Include agent decision results in the display summary
    print("\n=== Trust Game Experiment Summary ===")
    print(f"{'Sent':^10} | {'Tripled':^10} | {'Baseline':^20} | {'Intervention':^20}")
    print(
        f"{'Amount':^10} | {'Amount':^10} | {'Return':^10}{'Ratio':^10} | {'Return':^10}{'Ratio':^10}"
    )
    print("-" * 65)

    # Display agent decision results
    print(f"{'Agent Dec':^10} | {'Variable':^10} | {'':^20} | {'':^20}")
    print(
        f"{'Avg Sent:':^10} | {'':^10} | {agent_decision_results['avg_sent_baseline']:^10.2f}{'':^10} | {agent_decision_results['avg_sent_intervention']:^10.2f}{'':^10}"
    )

    # Display fixed amount results
    for result in results:
        print(
            f"{result['sent_amount']:^10} | {result['tripled_amount']:^10} | "
            f"{result['avg_return_baseline']:^10.2f}{result['return_ratio_baseline']:^10.2f} | "
            f"{result['avg_return_intervention']:^10.2f}{result['return_ratio_intervention']:^10.2f}"
        )

    return results, agent_decision_results


In [7]:
# Run the trust game experiments with different sent amounts
# Use the same agents and model setup
trust_game_results, agent_decision_results = run_trust_game_experiments(
    dummy_agents, model, start=10, end=100, step=10
)



--- Running Trust Game with agents deciding sent amounts ---

Running Trust Game (Agent Decision) without intervention...
--------------------------------
Sent amount: 50.0, Tripled amount: 150.0
--------------------------------

Running Trust Game (Agent Decision) with intervention...
--------------------------------
Sent amount: 100.0, Tripled amount: 300.0
--------------------------------

Average amounts sent by agents:
Baseline - Avg sent: 50.00
Intervention - Avg sent: 100.00
Error processing agent decision results: cannot access local variable 'results_dir' where it is not associated with a value

--- Running Trust Game with sent amount: 10 tokens ---

Running Trust Game (Sent: 10) without intervention...
Using fixed sent amount: 10, Tripled amount: 30

Running Trust Game (Sent: 10) with intervention...
Using fixed sent amount: 10, Tripled amount: 30

Sent amount: 10, Tripled amount: 30
Baseline - Avg return: 15.00, Return ratio: 0.50
Intervention - Avg return: 30.00, Return ra